In [ ]:
import numpy as np
import time
from pynq import Overlay
from pynq.pmbus import get_rails

BASE = '/home/xilinx/jupyter_notebooks/fpga-prosthetic-poc/'

# 전력 센서 초기화
rails = get_rails()
print('측정 가능한 전력 레일:')
for name, rail in rails.items():
    print(f'  {name}: {rail.power:.3f} W')

In [ ]:
def measure_power(func, n_trials=200):
    """함수 실행 중 평균 전력 측정 (W)"""
    power_samples = []
    for _ in range(n_trials):
        func()
        total = sum(r.power for r in rails.values())
        power_samples.append(total)
    return {
        'mean_w': np.mean(power_samples),
        'std_w':  np.std(power_samples),
        'min_w':  np.min(power_samples),
        'max_w':  np.max(power_samples),
    }

In [ ]:
# 유휴 전력 측정
idle_power = sum(r.power for r in rails.values())
print(f'유휴 전력: {idle_power:.3f} W')

In [ ]:
# ARM 추론 전력 측정
p = np.load(BASE + 'svm_params.npz')
scaler_mean = p['scaler_mean']
scaler_std  = p['scaler_std']
sv          = p['support_vectors']
dual_coef   = p['dual_coef']
intercept   = p['intercept']
gamma       = float(p['gamma'][0])
classes     = p['classes']
n_support   = p['n_support']
n_classes   = len(classes)
sv_start    = np.concatenate([[0], np.cumsum(n_support[:-1])])

X_test = np.random.randn(1, 96).astype(np.float32)

def arm_predict():
    X_scaled = (X_test - scaler_mean) / scaler_std
    diff = X_scaled[:, np.newaxis, :] - sv[np.newaxis, :, :]
    K = np.exp(-gamma * np.sum(diff ** 2, axis=2))
    votes = np.zeros((1, n_classes), dtype=np.int32)
    pair_idx = 0
    for i in range(n_classes):
        for j in range(i + 1, n_classes):
            si, ei = sv_start[i], sv_start[i] + n_support[i]
            sj, ej = sv_start[j], sv_start[j] + n_support[j]
            d = (K[:, si:ei] @ dual_coef[j-1, si:ei]
               + K[:, sj:ej] @ dual_coef[i, sj:ej]
               + intercept[pair_idx])
            votes[:, i] += (d > 0).astype(np.int32)
            votes[:, j] += (d <= 0).astype(np.int32)
            pair_idx += 1
    return classes[np.argmax(votes, axis=1)]

print('ARM 전력 측정 중...')
arm_result = measure_power(arm_predict)
print(f'[ARM 추론 전력]')
print(f'  평균: {arm_result["mean_w"]:.3f} W')
print(f'  최소: {arm_result["min_w"]:.3f} W')
print(f'  최대: {arm_result["max_w"]:.3f} W')

In [ ]:
# FPGA 추론 전력 측정
overlay = Overlay(BASE + 'svm_overlay.bit')
svm_ip = overlay.svm_inference_0
X_flat = X_test.flatten()

def fpga_predict():
    for i, val in enumerate(X_flat):
        svm_ip.write(0x10 + i * 4, int(np.float32(val).view(np.uint32)))
    svm_ip.write(0x00, 1)
    while not (svm_ip.read(0x00) & 0x2):
        pass
    return svm_ip.read(0x10 + 96 * 4)

print('FPGA 전력 측정 중...')
fpga_result = measure_power(fpga_predict)
print(f'[FPGA 추론 전력]')
print(f'  평균: {fpga_result["mean_w"]:.3f} W')
print(f'  최소: {fpga_result["min_w"]:.3f} W')
print(f'  최대: {fpga_result["max_w"]:.3f} W')

In [ ]:
# 최종 비교
print('=' * 45)
print('       ARM vs FPGA 최종 비교')
print('=' * 45)
print(f'유휴 전력:         {idle_power:.3f} W')
print(f'ARM 추론 전력:     {arm_result["mean_w"]:.3f} W')
print(f'FPGA 추론 전력:    {fpga_result["mean_w"]:.3f} W')
print()
arm_lat   = 24.580
fpga_lat  = 6.83
print(f'ARM 지연시간:      {arm_lat:.3f} ms')
print(f'FPGA 지연시간:     {fpga_lat:.3f} ms')
print(f'속도 향상:         {arm_lat/fpga_lat:.1f}x')
print(f'전력 비교:         {arm_result["mean_w"]/fpga_result["mean_w"]:.2f}x')
print('=' * 45)